# Notebook 01 — Data Validation

**This notebook is a gate.** Notebooks 02–04 assume the data has the properties asserted
here. If any check fails, downstream results are not trustworthy and should not be reported.

The purpose is not to demonstrate that the data is clean — it is not — but to establish
**exactly how it is dirty**, decide how each defect is handled, and record those decisions
where they can be audited.

## What is checked

| Section | Question |
|---|---|
| 2 | Did the nine raw tables load correctly? |
| 3 | Are the identifiers what they appear to be? |
| 4 | Do foreign keys resolve, and what is orphaned? |
| 5 | Are timestamps internally consistent? |
| 6 | Are values in plausible ranges? |
| 7 | Do the derived views exist and hold their grain? |
| 8 | Is repeat-purchase censoring handled correctly? |
| 9 | Do the layers reconcile against each other? |

Each check prints an expected value. Deviations are flagged rather than silently tolerated.

## 1 · Setup

In [ ]:
from google.colab import auth
auth.authenticate_user()

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from google.cloud import bigquery

PROJECT = "YOUR_PROJECT_ID"      # <- set this
DATASET = "olist"
client  = bigquery.Client(project=PROJECT)

def q(sql):
    return client.query(sql.replace("`olist.", f"`{PROJECT}.{DATASET}.")).to_dataframe()

RESULTS = []   # collected into the final scorecard

def check(name, actual, expected, tolerance=0):
    ok = abs(actual - expected) <= tolerance
    RESULTS.append({"check": name, "expected": expected, "actual": actual,
                    "status": "PASS" if ok else "FAIL"})
    print(f"[{'PASS' if ok else 'FAIL'}] {name:<46} expected {expected:>9,}  actual {actual:>9,}")
    return ok

print("ready")

---
# 2 · Raw layer — did the tables load?

Three defects in the published CSVs cause silent corruption if unhandled. All three were
encountered on this dataset and are documented in `data/README.md`.

In [ ]:
counts = q("""
SELECT 'orders' AS tbl, COUNT(*) AS n FROM `olist.orders` UNION ALL
SELECT 'customers',      COUNT(*) FROM `olist.customers` UNION ALL
SELECT 'order_items',    COUNT(*) FROM `olist.order_items` UNION ALL
SELECT 'order_payments', COUNT(*) FROM `olist.order_payments` UNION ALL
SELECT 'order_reviews',  COUNT(*) FROM `olist.order_reviews` UNION ALL
SELECT 'products',       COUNT(*) FROM `olist.products` UNION ALL
SELECT 'sellers',        COUNT(*) FROM `olist.sellers` UNION ALL
SELECT 'geolocation',    COUNT(*) FROM `olist.geolocation` UNION ALL
SELECT 'product_category_translation', COUNT(*) FROM `olist.product_category_translation`
""").set_index("tbl")

EXPECTED = {"orders":99441, "customers":99441, "order_items":112650,
            "order_payments":103886, "order_reviews":99224, "products":32951,
            "sellers":3095, "geolocation":1000163, "product_category_translation":71}

for t, e in EXPECTED.items():
    check(f"row count: {t}", int(counts.loc[t, "n"]), e)

**A table off by exactly +1 means its header row was ingested as data** — auto-detect failed
to recognise it. The symptom appears again in the next check as generic column names.

In [ ]:
generic = q("""
SELECT table_name, column_name, ordinal_position
FROM `olist.INFORMATION_SCHEMA.COLUMNS`
WHERE column_name LIKE 'string_field_%'
   OR column_name LIKE 'int64_field_%'
   OR column_name LIKE 'float64_field_%'
ORDER BY table_name, ordinal_position
""")

check("no generic column names", len(generic), 0)
if len(generic):
    print("\nTables loaded without a header row:")
    print(generic.to_string(index=False))
    print("\nFix: drop the table, re-upload with Auto-detect OFF and "
          "'Header rows to skip = 1', with an explicit schema.")

**The three load defects, for reference:**

1. **`order_reviews` — embedded newlines.** 3,852 reviews contain literal line breaks inside
   quoted comment fields. Valid RFC 4180, but BigQuery's default loader rejects it with
   *"Missing close quote character"*. Requires **Allow quoted newlines**.
2. **`product_category_translation` — UTF-8 BOM.** The byte-order mark is folded into the
   first column name, so joins fail with *"Name product_category_name not found"*.
3. **Header-row failure** — surfaces as `string_field_0`, and the row count is +1.

---
# 3 · Identifier integrity

**The single most consequential check in this notebook.** Olist ships two customer
identifiers and they mean different things.

In [ ]:
ids = q("""
SELECT
  COUNT(*)                           AS n_rows,
  COUNT(DISTINCT customer_id)        AS n_order_keys,
  COUNT(DISTINCT customer_unique_id) AS n_people
FROM `olist.customers`
""").iloc[0]

check("customers rows",          int(ids.n_rows),      99441)
check("distinct customer_id",    int(ids.n_order_keys), 99441)
check("distinct customer_unique_id", int(ids.n_people),  96096)

print(f"\ncustomer_id is unique per ROW  -> it is an ORDER key, not a person")
print(f"customer_unique_id identifies the PERSON")
print(f"difference = {int(ids.n_order_keys) - int(ids.n_people):,} -> these ARE the repeat customers")

In [ ]:
# Demonstrate the trap directly
trap = q("""
WITH by_order_key AS (
  SELECT customer_id AS k, COUNT(*) AS c FROM `olist.v_orders_enriched` GROUP BY customer_id
),
by_person AS (
  SELECT customer_unique_id AS k, COUNT(*) AS c FROM `olist.v_orders_enriched` GROUP BY customer_unique_id
)
SELECT 'customer_id (order key)' AS identifier, COUNT(*) AS entities,
       COUNTIF(c > 1) AS with_multiple, ROUND(100*COUNTIF(c>1)/COUNT(*), 2) AS repeat_pct
FROM by_order_key
UNION ALL
SELECT 'customer_unique_id (person)', COUNT(*), COUNTIF(c > 1),
       ROUND(100*COUNTIF(c>1)/COUNT(*), 2)
FROM by_person
""")
print(trap.to_string(index=False))

**Expected: `customer_id` yields a repeat rate of exactly 0.00%; `customer_unique_id` yields
3.12%.**

Keying customer analysis on `customer_id` makes every customer appear to have ordered once
*by construction*, producing a plausible-looking but entirely false result. All customer
analysis in this project uses `customer_unique_id`.

---
# 4 · Referential integrity

Which keys resolve, and what is legitimately orphaned.

In [ ]:
orphans = q("""
SELECT
  (SELECT COUNT(*) FROM `olist.order_items`    i
     LEFT JOIN `olist.orders`    o USING (order_id)    WHERE o.order_id    IS NULL) AS items_no_order,
  (SELECT COUNT(*) FROM `olist.order_items`    i
     LEFT JOIN `olist.products`  p USING (product_id)  WHERE p.product_id  IS NULL) AS items_no_product,
  (SELECT COUNT(*) FROM `olist.order_items`    i
     LEFT JOIN `olist.sellers`   s USING (seller_id)   WHERE s.seller_id   IS NULL) AS items_no_seller,
  (SELECT COUNT(*) FROM `olist.orders`         o
     LEFT JOIN `olist.customers` c USING (customer_id) WHERE c.customer_id IS NULL) AS orders_no_customer,
  (SELECT COUNT(*) FROM `olist.orders` o
     WHERE NOT EXISTS (SELECT 1 FROM `olist.order_items`    i WHERE i.order_id = o.order_id)) AS orders_no_items,
  (SELECT COUNT(*) FROM `olist.orders` o
     WHERE NOT EXISTS (SELECT 1 FROM `olist.order_reviews`  r WHERE r.order_id = o.order_id)) AS orders_no_review,
  (SELECT COUNT(*) FROM `olist.orders` o
     WHERE NOT EXISTS (SELECT 1 FROM `olist.order_payments` p WHERE p.order_id = o.order_id)) AS orders_no_payment
""").iloc[0]

for k in ["items_no_order","items_no_product","items_no_seller","orders_no_customer"]:
    check(f"no orphans: {k}", int(orphans[k]), 0)

print()
check("orders with no items",   int(orphans.orders_no_items),   775)
check("orders with no review",  int(orphans.orders_no_review),  768)
check("orders with no payment", int(orphans.orders_no_payment), 1)

**All foreign keys resolve — no true orphans.** The three non-zero counts are legitimate
absences, not integrity failures:

- **775 orders with no items** — predominantly cancelled or unavailable. These are dropped by
  the `INNER JOIN` in `v_fact_orders`, which is why the base layer holds 98,666 rows rather
  than 99,441.
- **768 orders with no review** — customers who did not respond. Excluded from review
  analysis; not imputed.
- **1 order with no payment** — a single anomaly, immaterial.

In [ ]:
dupes = q("""
SELECT
  (SELECT COUNT(*) FROM (SELECT order_id FROM `olist.order_reviews`
      GROUP BY order_id HAVING COUNT(*) > 1))  AS orders_multi_review,
  (SELECT COUNT(*) FROM (SELECT review_id FROM `olist.order_reviews`
      GROUP BY review_id HAVING COUNT(*) > 1)) AS duplicated_review_ids,
  (SELECT COUNT(*) FROM (SELECT order_id FROM `olist.order_payments`
      GROUP BY order_id HAVING COUNT(*) > 1))  AS orders_multi_payment
""").iloc[0]

check("orders with >1 review",  int(dupes.orders_multi_review),  547)
check("orders with >1 payment", int(dupes.orders_multi_payment), 2961)
print(f"\nreview_id values appearing on multiple rows: {int(dupes.duplicated_review_ids):,}")

**Expected: 547 orders with multiple reviews, 2,961 with split payments.**

Note the third figure separately: **`review_id` is not unique** — some review identifiers
appear against more than one order. `review_id` is therefore *not* a usable primary key.
`v_reviews_clean` deduplicates on `order_id`, keeping the most recent review, which sidesteps
the problem.

Split payments are rolled up to an order total in `v_payments_order`, with the dominant
method taken as the largest single line.

---
# 5 · Timestamp consistency

The order pipeline has a logical sequence: purchase → approved → carrier → delivered.
Violations indicate data-entry problems and constrain which pipeline-decomposition analyses
are trustworthy.

In [ ]:
ts = q("""
SELECT
  COUNTIF(order_status='delivered' AND order_delivered_customer_date IS NULL) AS delivered_no_date,
  COUNTIF(order_delivered_customer_date < order_purchase_timestamp)          AS delivered_before_purchase,
  COUNTIF(order_approved_at            < order_purchase_timestamp)           AS approved_before_purchase,
  COUNTIF(order_delivered_carrier_date < order_approved_at)                  AS carrier_before_approved,
  COUNTIF(order_delivered_customer_date< order_delivered_carrier_date)       AS delivered_before_carrier,
  COUNTIF(order_approved_at IS NULL)                                          AS null_approved,
  COUNTIF(order_delivered_carrier_date IS NULL)                               AS null_carrier
FROM `olist.orders`
""").iloc[0]

check("delivered before purchase", int(ts.delivered_before_purchase), 0)
check("approved before purchase",  int(ts.approved_before_purchase),  0)
print()
for k, e in [("delivered_no_date", 8), ("carrier_before_approved", 1359),
             ("delivered_before_carrier", 23)]:
    print(f"  {k:<28} {int(ts[k]):>7,}   (expected ~{e:,})")

**Two impossible orderings are absent** — nothing is delivered or approved before it was
purchased. Good.

**Three anomalies are present and must be handled:**

- **8 orders marked delivered with no delivery date.** Excluded from delivery analysis by the
  `delivered_ts IS NOT NULL` filter carried throughout.
- **1,359 orders where the carrier received the parcel before payment approval.** Roughly
  1.4% of orders. Plausibly a batch-processing artefact where approval timestamps are written
  late. **This is the constraint on pipeline-leg decomposition:** the
  `approved → carrier` leg is negative for these rows, so any "where does the time go"
  analysis must either exclude them or report medians rather than means.
- **23 orders delivered to the customer before the carrier collected them.** Clearly
  erroneous; immaterial at this volume but should be excluded from leg analysis.

Total delivery time (purchase → delivered) is unaffected by all three and remains reliable.

---
# 6 · Value ranges

In [ ]:
vals = q("""
SELECT
  (SELECT COUNTIF(price <= 0)          FROM `olist.order_items`)    AS price_nonpositive,
  (SELECT COUNTIF(freight_value < 0)   FROM `olist.order_items`)    AS freight_negative,
  (SELECT COUNTIF(freight_value = 0)   FROM `olist.order_items`)    AS freight_zero,
  (SELECT COUNTIF(payment_value <= 0)  FROM `olist.order_payments`) AS payment_nonpositive,
  (SELECT COUNTIF(payment_installments = 0) FROM `olist.order_payments`) AS installments_zero,
  (SELECT COUNTIF(product_category_name IS NULL) FROM `olist.products`)  AS null_category,
  (SELECT COUNTIF(product_weight_g IS NULL)      FROM `olist.products`)  AS null_weight
""").iloc[0]

check("no non-positive item prices", int(vals.price_nonpositive), 0)
check("no negative freight",         int(vals.freight_negative),  0)
print()
for k, e in [("freight_zero", 383), ("payment_nonpositive", 9),
             ("installments_zero", 2), ("null_category", 610), ("null_weight", 2)]:
    print(f"  {k:<24} {int(vals[k]):>6,}   (expected {e:,})")

In [ ]:
dist = q("""
SELECT
  MIN(delivery_days)                                          AS min_days,
  APPROX_QUANTILES(delivery_days, 100)[OFFSET(1)]             AS p1,
  APPROX_QUANTILES(delivery_days, 100)[OFFSET(50)]            AS median,
  APPROX_QUANTILES(delivery_days, 100)[OFFSET(99)]            AS p99,
  MAX(delivery_days)                                          AS max_days,
  COUNTIF(delivery_days = 0)                                  AS same_day,
  COUNTIF(delivery_days > 60)                                 AS over_60d,
  COUNT(*)                                                    AS n
FROM `olist.v_fact_orders`
WHERE order_status = 'delivered' AND delivery_days IS NOT NULL
""").iloc[0]

print("delivery_days distribution")
print(f"  min {int(dist.min_days)} | p1 {int(dist.p1)} | median {int(dist.median)} "
      f"| p99 {int(dist.p99)} | max {int(dist.max_days)}")
print(f"  same-day ({int(dist.same_day)}) and >60 days ({int(dist.over_60d)}, "
      f"{100*dist.over_60d/dist.n:.2f}%)")

**Expected: min 0, median 10, p99 46, max 209 days.**

**Decision on outliers: retain them.** The 288 orders exceeding 60 days (0.30%) are extreme
but not implausible for Brazilian long-distance logistics, and they are precisely the
pathological fulfilment events that Notebooks 03 and 04 are about. Trimming them would remove
the phenomenon under study. Robust statistics (medians, non-parametric tests) are used instead
of winsorising.

**383 items with zero freight** are plausible — free-shipping promotions. Retained, but they
inflate the freight-to-value ratio's denominator behaviour and are worth noting where that
metric appears.

---
# 7 · View layer and grain

The gate for Notebooks 02–04.

In [ ]:
views = q("""
SELECT table_name FROM `olist.INFORMATION_SCHEMA.VIEWS` ORDER BY table_name
""")

REQUIRED = ["v_orders_clean", "v_customers_clean", "v_orders_enriched",
            "v_payments_order", "v_reviews_clean", "v_products_clean",
            "v_geo_dedup", "v_order_items_clean", "v_fact_order_items",
            "v_order_items_agg", "v_customer_order_seq", "v_fact_orders"]

present = set(views.table_name)
for v in REQUIRED:
    status = "PASS" if v in present else "FAIL"
    RESULTS.append({"check": f"view exists: {v}", "expected": 1,
                    "actual": int(v in present), "status": status})
    print(f"[{status}] view exists: {v}")

missing = [v for v in REQUIRED if v not in present]
if missing:
    print(f"\nMissing: {missing}")
    print("Run sql/01_cleaning_views.sql then sql/02_order_grain_base.sql")

In [ ]:
grain = q("""
SELECT
  COUNT(*)                            AS n_rows,
  COUNT(DISTINCT order_id)            AS n_distinct_orders,
  COUNTIF(order_status = 'delivered') AS n_delivered,
  COUNTIF(n_items > 1)                AS n_multi_item
FROM `olist.v_fact_orders`
""").iloc[0]

check("v_fact_orders rows",            int(grain.n_rows),           98666)
check("v_fact_orders distinct orders", int(grain.n_distinct_orders), 98666)
check("v_fact_orders delivered",       int(grain.n_delivered),      96478)
check("multi-item orders",             int(grain.n_multi_item),      9803)

assert int(grain.n_rows) == int(grain.n_distinct_orders), \
    "GRAIN FAILURE: v_fact_orders has duplicate order_id - check the geolocation join"
print("\nGRAIN OK - one row per order")

**This assertion is the gate.** If `n_rows` exceeds `n_distinct_orders`, something is fanning
out — most likely a raw join to `geolocation`, which carries ~52.6 rows per zip prefix and
must go through `v_geo_dedup`. A ~50× fan-out would not be subtle, but a partial one might be.

**9,803 multi-item orders (9.9%)** is the number that makes grain discipline matter. Reading
order-level outcomes from the item layer weights those orders by basket size.

In [ ]:
nulls = q("""
SELECT
  COUNTIF(review_score      IS NULL) AS no_review,
  COUNTIF(delivery_days     IS NULL) AS no_delivery_date,
  COUNTIF(max_distance_km   IS NULL) AS no_distance,
  COUNTIF(category          IS NULL) AS no_category,
  COUNTIF(order_payment_total IS NULL) AS no_payment,
  COUNT(*)                           AS n
FROM `olist.v_fact_orders`
WHERE order_status = 'delivered'
""").iloc[0]

print("Nulls among DELIVERED orders (the modelling population):\n")
for k in ["no_review","no_delivery_date","no_distance","no_category","no_payment"]:
    print(f"  {k:<20}{int(nulls[k]):>7,}   ({100*nulls[k]/nulls.n:.2f}%)")

**Missingness is handled by exclusion, never imputation**, because each null is informative:
a missing review means the customer chose not to respond, and imputing a score would
manufacture data. Each analysis states its own effective *n*.

---
# 8 · Censoring

Verifying that the three-state repeat flags behave as designed. This is what makes the
retention numbers in Notebook 03 defensible.

In [ ]:
cens = q("""
SELECT '30d'  AS window_label, COUNTIF(repeat_within_30d  IS NOT NULL) AS usable, COUNT(*) AS total
FROM `olist.v_fact_orders` WHERE is_first_order
UNION ALL SELECT '60d',  COUNTIF(repeat_within_60d  IS NOT NULL), COUNT(*)
FROM `olist.v_fact_orders` WHERE is_first_order
UNION ALL SELECT '90d',  COUNTIF(repeat_within_90d  IS NOT NULL), COUNT(*)
FROM `olist.v_fact_orders` WHERE is_first_order
UNION ALL SELECT '180d', COUNTIF(repeat_within_180d IS NOT NULL), COUNT(*)
FROM `olist.v_fact_orders` WHERE is_first_order
""")
cens["pct_usable"] = (100 * cens.usable / cens.total).round(1)
print(cens.to_string(index=False))

**Expected: 100.0% / 98.5% / 90.5% / 71.7%.**

The decline is the censoring working. A customer acquired days before the data ends
(2018-10-17) cannot have had 90 days of opportunity to return, so their 90-day flag is NULL
and they are excluded from that denominator — rather than being counted as someone who chose
not to return.

**90 days is adopted as the headline window**: long enough to capture most repeat behaviour,
while retaining 90.5% of first orders.

In [ ]:
# The flags must be logically consistent
consist = q("""
SELECT
  COUNTIF(repeat_within_30d = TRUE AND repeat_within_90d = FALSE)   AS impossible_30_not_90,
  COUNTIF(repeat_within_90d = TRUE AND repeat_within_180d = FALSE)  AS impossible_90_not_180,
  COUNTIF(repeat_within_30d IS NOT NULL AND exposure_days < 30)     AS flag_without_exposure,
  COUNTIF(days_to_next_order < 0)                                    AS negative_gap
FROM `olist.v_fact_orders` WHERE is_first_order
""").iloc[0]

for k in ["impossible_30_not_90","impossible_90_not_180","flag_without_exposure","negative_gap"]:
    check(f"consistency: {k}", int(consist[k]), 0)

**All four must be zero.** A repeat within 30 days is necessarily a repeat within 90; a flag
cannot be set without sufficient exposure; and the gap to the next order cannot be negative.
These catch logic errors in the `CASE` expressions that row counts alone would miss.

In [ ]:
rates = q("""
SELECT
  ROUND(100 * AVG(CAST(ever_repeated      AS INT64)), 2) AS ever_uncensored,
  ROUND(100 * AVG(CAST(repeat_within_30d  AS INT64)), 2) AS r30,
  ROUND(100 * AVG(CAST(repeat_within_60d  AS INT64)), 2) AS r60,
  ROUND(100 * AVG(CAST(repeat_within_90d  AS INT64)), 2) AS r90,
  ROUND(100 * AVG(CAST(repeat_within_180d AS INT64)), 2) AS r180
FROM `olist.v_fact_orders` WHERE is_first_order
""").iloc[0]
print(rates.to_frame("pct").to_string())

**Expected: ever 3.12% | 30d 1.60% | 60d 1.96% | 90d 2.26% | 180d 3.05%.**

The windowed rates are *lower* than the uncensored figure, which is correct and worth stating
plainly: each is a shorter window, not an upward correction. What the correction buys is
**comparability** — every customer counted in the 90-day rate had 90 days of opportunity,
whereas the 3.12% mixes customers with 21 months and customers with five days.

`AVG` ignores NULLs in BigQuery, so each window is automatically computed on its eligible
population. That is the entire mechanism.

---
# 9 · Cross-layer reconciliation

The layers must agree with each other, or an aggregation is wrong somewhere.

In [ ]:
rec = q("""
SELECT
  (SELECT ROUND(SUM(price), 2)        FROM `olist.v_fact_order_items`) AS item_layer_value,
  (SELECT ROUND(SUM(item_value), 2)   FROM `olist.v_fact_orders`)      AS order_layer_value,
  (SELECT COUNT(*)                    FROM `olist.v_fact_order_items`) AS item_rows,
  (SELECT SUM(n_items)                FROM `olist.v_fact_orders`)      AS order_layer_items
""").iloc[0]

gap = rec.item_layer_value - rec.order_layer_value
print(f"item value, item layer  : {rec.item_layer_value:>15,.2f}")
print(f"item value, order layer : {rec.order_layer_value:>15,.2f}")
print(f"difference              : {gap:>15,.2f}\n")
print(f"item rows               : {int(rec.item_rows):>15,}")
print(f"sum(n_items) at order   : {int(rec.order_layer_items):>15,}")
print("\nSmall differences are rounding (order layer rounds per order). "
      "Large ones mean an aggregation is wrong.")

---
# 10 · Validation scorecard

In [ ]:
scorecard = pd.DataFrame(RESULTS)
n_fail = (scorecard.status == "FAIL").sum()

print(scorecard.to_string(index=False))
print(f"\n{'='*70}")
if n_fail == 0:
    print("ALL CHECKS PASSED - downstream notebooks may proceed")
else:
    print(f"{n_fail} CHECK(S) FAILED - resolve before running Notebooks 02-04")
    print(scorecard[scorecard.status == "FAIL"].to_string(index=False))

---
# 11 · Data-quality decisions

Every defect found above, and how it is handled. This table is the audit trail; it mirrors
`docs/methodology.md`.

| Issue | Scale | Decision |
|---|---|---|
| `customer_id` unique per order | 99,441 keys vs 96,096 people | All customer analysis keys on `customer_unique_id` |
| Orders with no items | 775 | Excluded via INNER JOIN in `v_fact_orders` |
| Orders with no review | 768 | Excluded from review analysis; not imputed |
| Orders with >1 review | 547 | Deduplicated to most recent |
| `review_id` not unique | 814 values repeat | `review_id` not used as a key; dedup on `order_id` |
| Split payments | 2,961 orders | Rolled to order total; dominant method = largest line |
| Geolocation duplication | ~52.6 rows/zip | Deduplicated to one centroid before any join |
| Delivered with no delivery date | 8 | Excluded from delivery analysis |
| Carrier before approval | 1,359 (1.4%) | **Constrains pipeline-leg analysis**; total delivery time unaffected |
| Delivered before carrier | 23 | Excluded from leg analysis |
| Zero freight | 383 items | Retained (free-shipping promotions) |
| Null product category | 610 | Coded `uncategorised`, not dropped |
| Delivery > 60 days | 288 (0.30%) | **Retained** — these are the events under study |
| Embedded newlines in reviews | 3,852 | Collapsed to spaces at load |
| Sparse period ends | — | Time series trimmed; trimming stated where applied |

## What this notebook establishes

The data is usable, with three constraints carried forward:

1. **Grain must be enforced.** 9.9% of orders are multi-item; order-level outcomes come from
   `v_fact_orders` only.
2. **Repeat purchase must be censoring-aware.** The raw 3.12% is not retention.
3. **Pipeline-leg decomposition is limited** by 1,359 orders with inconsistent approval and
   carrier timestamps. Total delivery time is unaffected.